In [40]:
import sys
import os
sys.path.append(os.path.abspath('../'))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from functions.preproc import *
from functions.svm import *

from sklearn.model_selection import train_test_split
from sksurv.svm  import FastSurvivalSVM
from sksurv.metrics import concordance_index_censored
from sksurv.util import Surv
from sksurv.metrics import integrated_brier_score
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [41]:
dataset = pd.read_csv("data_treated.csv")

treatment_variables = []
with open("diagnosis.txt", "r") as file:
    for line in file:
        treatment_variables.append(line.strip())

diagnosis_variables = []
with open("treatment.txt", "r") as file:
    for line in file:
        diagnosis_variables.append(line.strip())

survival_dataset = dataset.drop('Unnamed: 0', axis=1)

feature_columns = treatment_variables + diagnosis_variables

Did not transformed ki like I did earlier

In [42]:
data_x = survival_dataset[feature_columns]
data_y = Surv.from_arrays(
    event=survival_dataset['Regional LN recurrence'],
    time=survival_dataset['Survival days']
)
groups = survival_dataset['PERSON_ID']
event_indicator = survival_dataset['Regional LN recurrence']

time = data_y["time"]
valid_mask = time > 0

data_x = data_x[valid_mask]
data_y = data_y[valid_mask]
groups = groups[valid_mask]
event_indicator = event_indicator[valid_mask]

In [50]:
sgkf = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=12)
cv = sgkf.split(X=data_x, y=event_indicator, groups=groups)

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", FastSurvivalSVM(random_state=12))
])

small_param_grid = {
    "svm__alpha": [1],
    "svm__rank_ratio": [0.1]
}

param_grid = {
    "svm__alpha": [0.01, 0.1, 1, 10],              # Regularization
    "svm__rank_ratio": [0.1, 0.5, 1.0],            # Pairwise ranking
    "svm__max_iter": [5000, 10000],                # Convergence control
    "svm__tol": [1e-4, 1e-5]                       # Precision of convergence
}

# Best parameters: {'svm__alpha': 0.01, 'svm__max_iter': 5000, 'svm__rank_ratio': 1.0, 'svm__tol': 0.0001}


In [51]:
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=cv,
    scoring=score_survival_model,
    n_jobs=-1,
    verbose=1
)

In [52]:
grid_search.fit(data_x, data_y)

Fitting 4 folds for each of 48 candidates, totalling 192 fits


GridSearchCV(cv=<generator object _BaseKFold.split at 0x00000257906028C0>,
             estimator=Pipeline(steps=[('scaler', StandardScaler()),
                                       ('svm',
                                        FastSurvivalSVM(random_state=12))]),
             n_jobs=-1,
             param_grid={'svm__alpha': [0.01, 0.1, 1, 10],
                         'svm__max_iter': [5000, 10000],
                         'svm__rank_ratio': [0.1, 0.5, 1.0],
                         'svm__tol': [0.0001, 1e-05]},
             scoring=<function score_survival_model at 0x000002579A46C900>,
             verbose=1)

In [53]:
print("Best parameters:", grid_search.best_params_)
print("Best C-Index:", grid_search.best_score_)
best_model = grid_search.best_estimator_

Best parameters: {'svm__alpha': 0.01, 'svm__max_iter': 5000, 'svm__rank_ratio': 1.0, 'svm__tol': 0.0001}
Best C-Index: 0.7566241206692295


In [57]:
results = train_evaluate(data_x, data_y, groups, sgkf, best_model)
results

,fold,c_index
0,0,0.711358
1,1,0.879668
2,2,0.811366
3,3,0.624104
4,mean,0.756624
5,std,0.112171
